# BeyondBench: Creating Custom Tasks

This notebook shows how to create your own evaluation task by extending `BaseTask`.

**What you'll learn:**
- The structure of `BaseTask` and the four abstract members to implement
- How to generate data dynamically (contamination-resistant)
- How to write robust answer parsers
- How to register and run your custom task

**Prerequisites:**
```bash
pip install beyondbench
```

## 1. Understanding BaseTask

Every BeyondBench task extends `BaseTask` and implements four members:

| Member | Type | Purpose |
|--------|------|---------|
| `task_name` | property | Unique string identifier |
| `generate_data(**kwargs)` | method | Produce evaluation data points |
| `create_prompt(data_point)` | method | Convert a data point to a prompt |
| `evaluate_response(response, data_point)` | method | Check if the answer is correct |

In [ ]:
from beyondbench.core.base_task import BaseTask
import inspect

# View the abstract interface
abstract_methods = [
    name for name, obj in inspect.getmembers(BaseTask)
    if getattr(obj, "__isabstractmethod__", False)
]
print("Abstract members in BaseTask:", abstract_methods)

## 2. Example 1: Digit Sum Task

A simple task: given a random integer, compute the sum of its digits.

In [ ]:
import random
import re
from beyondbench.core.base_task import BaseTask


class DigitSumTask(BaseTask):
    """Evaluate model ability to sum the digits of an integer."""

    @property
    def task_name(self) -> str:
        return "digit_sum"

    def generate_data(self, **kwargs):
        """Generate random integers between 100 and 99999."""
        if self.seed is not None:
            random.seed(self.seed)
        return [random.randint(100, 99999) for _ in range(self.num_samples)]

    def create_prompt(self, data_point: int) -> str:
        """Ask the model to sum the digits."""
        return (
            f"What is the sum of the digits of the number {data_point}?\n\n"
            r"Put your final answer in \boxed{answer}."
        )

    def evaluate_response(self, response: str, data_point: int) -> bool:
        """Check if the parsed answer equals the correct digit sum."""
        expected = sum(int(d) for d in str(abs(data_point)))

        # Try \boxed{} notation first
        match = re.search(r'\\boxed\{([^}]+)\}', response)
        if match:
            try:
                return int(match.group(1).strip()) == expected
            except ValueError:
                pass

        # Fallback: last number in response
        numbers = re.findall(r'\b(\d+)\b', response)
        if numbers:
            try:
                return int(numbers[-1]) == expected
            except ValueError:
                pass

        return False


print("DigitSumTask defined successfully")

In [ ]:
# Test the task logic without a real model
from unittest.mock import MagicMock

mock_handler = MagicMock()
mock_handler.get_model_info.return_value = {"model_name": "test", "backend": "mock"}
mock_handler.generate.return_value = [r"The digit sum is \boxed{15}"]

task = DigitSumTask(
    model_handler=mock_handler,
    output_dir="/tmp/digit_sum_test",
    min_val=1,
    max_val=1000,
    num_folds=1,
    num_samples=5,
    store_details=False,
    temperature=0.0,
    top_p=1.0,
    max_tokens=512,
    seed=42,
)

print("Task name:", task.task_name)

# Generate some data
data = task.generate_data()
print("Generated data (5 samples):", data[:5])

# Test prompt creation
print("\nSample prompt:")
print(task.create_prompt(12345))

# Test evaluation
correct_response = r"\boxed{15}"  # sum of 1+2+3+4+5
wrong_response = r"\boxed{10}"
print("\nCorrect response evaluated as:", task.evaluate_response(correct_response, 12345))
print("Wrong response evaluated as:  ", task.evaluate_response(wrong_response, 12345))

## 3. Example 2: Count Vowels Task

A more complex example that works with strings rather than numbers.

In [ ]:
import random
import string
import re
from beyondbench.core.base_task import BaseTask


class CountVowelsTask(BaseTask):
    """Count the number of vowels (a, e, i, o, u) in a random word sequence."""

    WORD_LIST = [
        "algorithm", "benchmark", "evaluation", "language", "reasoning",
        "computation", "mathematics", "sequence", "optimization", "neural",
        "transformer", "attention", "prediction", "inference", "accuracy",
    ]

    @property
    def task_name(self) -> str:
        return "count_vowels"

    def generate_data(self, num_words: int = 3, **kwargs):
        """Generate random 3-word phrases."""
        if self.seed is not None:
            random.seed(self.seed)
        data = []
        for _ in range(self.num_samples):
            words = random.sample(self.WORD_LIST, min(num_words, len(self.WORD_LIST)))
            phrase = " ".join(words)
            data.append(phrase)
        return data

    def create_prompt(self, data_point: str) -> str:
        return (
            f'How many vowels (a, e, i, o, u) appear in the phrase: "{data_point}"?\n\n'
            "Count both uppercase and lowercase vowels.\n\n"
            r"Put your final answer in \boxed{answer}."
        )

    def evaluate_response(self, response: str, data_point: str) -> bool:
        expected = sum(1 for c in data_point.lower() if c in 'aeiou')

        match = re.search(r'\\boxed\{([^}]+)\}', response)
        if match:
            try:
                return int(match.group(1).strip()) == expected
            except ValueError:
                pass
        return False


# Quick sanity check
phrase = "algorithm benchmark"
expected = sum(1 for c in phrase if c in 'aeiou')
print(f"Phrase: '{phrase}' -> vowel count: {expected}")
print("CountVowelsTask defined successfully")

## 4. Registering and Running a Custom Task

Once you have a task class, register it with `TaskRegistry` and run it with `EvaluationEngine`.

In [ ]:
from beyondbench.core.task_registry import TaskRegistry

# Create a registry and register our custom tasks
registry = TaskRegistry()
registry.register_task("digit_sum", DigitSumTask, "easy")
registry.register_task("count_vowels", CountVowelsTask, "easy")

# Verify registration
print("digit_sum registered:", registry.validate_task_name("digit_sum"))
print("count_vowels registered:", registry.validate_task_name("count_vowels"))

# Retrieve class
TaskClass = registry.get_task_class("digit_sum")
print("Retrieved class:", TaskClass.__name__)

In [ ]:
import os

# Run with a real model if API key available
api_key = os.environ.get("OPENAI_API_KEY", "")

if api_key:
    from beyondbench.models.model_handler import ModelHandler
    from beyondbench.core.evaluation_engine import EvaluationEngine

    handler = ModelHandler(
        model_id="gpt-4o",
        api_provider="openai",
        api_key=api_key,
    )

    engine = EvaluationEngine(
        model_handler=handler,
        output_dir="/tmp/bb_custom_tasks",
    )

    # The engine uses its own registry, so we need to add our task to it
    engine.task_registry.register_task("digit_sum", DigitSumTask, "easy")

    results = engine.run_evaluation(
        suite="easy",
        tasks=["digit_sum"],
        datapoints=10,
        seed=42,
    )

    task_result = results["task_results"].get("digit_sum", {})
    if isinstance(task_result, list) and task_result:
        acc = sum(m.get("accuracy", 0) for m in task_result) / len(task_result)
    elif isinstance(task_result, dict):
        acc = task_result.get("summary", {}).get("avg_accuracy", 0)
    else:
        acc = 0
    print(f"digit_sum accuracy: {acc:.1%}")
else:
    print("Set OPENAI_API_KEY to run the custom task with a real model")

## 5. Best Practices for Custom Tasks

### Contamination Resistance
Always generate data randomly from a seed — never hardcode examples.

### Parser Robustness
Support multiple answer formats:

In [ ]:
def robust_parse_integer(response: str, expected: int) -> bool:
    """
    Robust integer parser that handles multiple common response formats:
    - \\boxed{42}
    - 'The answer is 42.'
    - '42' (bare number on last line)
    """
    import re

    # Strategy 1: \boxed{} notation
    boxed = re.search(r'\\boxed\{([^}]+)\}', response)
    if boxed:
        try:
            return int(boxed.group(1).strip()) == expected
        except ValueError:
            pass

    # Strategy 2: "answer is X" pattern
    answer_is = re.search(r'(?:answer|result|sum|total)[^\d]*?(-?\d+)', response, re.IGNORECASE)
    if answer_is:
        try:
            return int(answer_is.group(1)) == expected
        except ValueError:
            pass

    # Strategy 3: last number in response
    all_numbers = re.findall(r'-?\d+', response)
    if all_numbers:
        try:
            return int(all_numbers[-1]) == expected
        except ValueError:
            pass

    return False


# Test the parser
test_cases = [
    (r"The digit sum is \boxed{15}", 15, True),
    ("The answer is 15.", 15, True),
    ("After computing: 15", 15, True),
    (r"\boxed{10}", 15, False),
]

print("Parser tests:")
for response, expected, expected_result in test_cases:
    result = robust_parse_integer(response, expected)
    status = "PASS" if result == expected_result else "FAIL"
    print(f"  [{status}] '{response[:40]}...' -> {result}")

## 6. Adding Your Task to the Package

To permanently add your task to BeyondBench:

1. Save `my_task.py` to `beyondbench/tasks/easy/my_task_task.py`
2. Add `'my_task'` to `easy_tasks` in `beyondbench/core/task_registry.py`
3. Export from `beyondbench/tasks/easy/__init__.py`
4. Write tests in `tests/unit/tasks/test_my_task_task.py`
5. Verify with `beyondbench list-tasks --suite easy`

See [contributing.md](../contributing.md) for the full guide.

In [ ]:
# Verify your task would appear in the task list
from beyondbench.core.task_registry import TaskRegistry

registry = TaskRegistry()
registry.register_task("digit_sum", DigitSumTask, "easy")

easy_tasks = registry.get_tasks_for_suite("easy")
print(f"Easy tasks ({len(easy_tasks)} total):")
print("  digit_sum included:", "digit_sum" in easy_tasks)

## Summary

You've learned how to:
- Extend `BaseTask` with `task_name`, `generate_data`, `create_prompt`, and `evaluate_response`
- Generate data dynamically for contamination resistance
- Write robust parsers with multiple fallback strategies
- Register a custom task with `TaskRegistry`

**Next:** `03_model_comparison.ipynb` — Compare multiple models on the same task suite.